# Data Preparation and Exploratory Data Analysis

In [ ]:
# Import necessary libraries and set visualization style
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

In [ ]:
#this section is for google colab. If you are running this code on your local machine, you can ignore this section and directly load the dataset from your local directory.

# Mount Google Drive to access datasets
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#this also is for google colab. If you are running this code on your local machine, you can ignore this section and directly load the dataset from your local directory.

# Load MovieLens datasets (ratings and movies) from Google Drive
ratings_file_path = '/content/drive/MyDrive/DataSetMovieLens/ratings.csv'
movies_file_path = '/content/drive/MyDrive/DataSetMovieLens/movies.csv'
tags_file_path = '/content/drive/MyDrive/DataSetMovieLens/tags.csv'

# Load the datasets (here you can also load from local directory if not using Google Colab just by providing the path to the files)
ratings = pd.read_csv(ratings_file_path)
movies = pd.read_csv(movies_file_path)
tags = pd.read_csv(tags_file_path)

# Preview data
print("Ratings Dataset")
display(ratings.head())

print("\nMovies Dataset")
display(movies.head())

print("\nTags Dataset")
display(tags.head())

In [ ]:
# Get initial overview of data shapes and types for both datasets
print("Ratings Shape:", ratings.shape)
print("Movies Shape:", movies.shape)

print("\nRatings Info")
ratings.info()

print("\nMovies Info")
movies.info()


In [ ]:
# Calculate and display the number of unique users and movies
num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()

print("Number of unique users:", num_users)
print("Number of unique movies:", num_movies)

In [ ]:
tags_per_movie = tags.groupby('movieId').size()

plt.figure()
sns.histplot(tags_per_movie, bins=30)
plt.title("Distribution of Tags per Movie")
plt.xlabel("Number of Tags")
plt.ylabel("Movies")
plt.show()

In [ ]:
# Visualize the distribution of movie ratings
plt.figure()
sns.countplot(x="rating", data=ratings)
plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

In [ ]:
# Identify and visualize the top 10 most rated movies
movie_ratings_count = ratings.groupby('movieId').size().sort_values(ascending=False)

top_movies = movie_ratings_count.head(10)

top_movies = top_movies.reset_index()
top_movies.columns = ['movieId', 'rating_count']

# Merge with movie titles
top_movies = top_movies.merge(movies, on='movieId')

print("Top 10 Most Rated Movies")
display(top_movies[['title', 'rating_count']])

# Plot
plt.figure()
sns.barplot(y='title', x='rating_count', data=top_movies)
plt.title("Top 10 Most Rated Movies")
plt.xlabel("Number of Ratings")
plt.ylabel("Movie Title")
plt.show()

In [ ]:
# Calculate and display the sparsity of the user-item matrix
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')

num_possible_ratings = user_item_matrix.shape[0] * user_item_matrix.shape[1]
num_actual_ratings = ratings.shape[0]

sparsity = (1 - (num_actual_ratings / num_possible_ratings)) * 100

print("User-Item Matrix Shape:", user_item_matrix.shape)
print("Matrix Sparsity: {:.2f}%".format(sparsity))

In [ ]:
# Visualize the distribution of ratings per user
ratings_per_user = ratings.groupby('userId').size()

plt.figure()
sns.histplot(ratings_per_user, bins=30, kde=True)
plt.title("Distribution of Ratings per User")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Users")
plt.show()

In [ ]:
# Visualize the distribution of ratings per movie
ratings_per_movie = ratings.groupby('movieId').size()

plt.figure()
sns.histplot(ratings_per_movie, bins=30, kde=True)
plt.title("Distribution of Ratings per Movie")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Movies")
plt.show()

In [ ]:
# Identify and display the top 10 highest-rated movies
movie_avg_rating = ratings.groupby('movieId')['rating'].mean()

movie_avg_rating = movie_avg_rating.reset_index()
movie_avg_rating = movie_avg_rating.merge(movies, on='movieId')

top_rated = movie_avg_rating.sort_values(by='rating', ascending=False).head(10)

print("Top 10 Highest Rated Movies")
display(top_rated[['title','rating']])

In [ ]:
# Explore and visualize the distribution of movie genres
genres = movies['genres'].str.split('|').explode()

plt.figure()
sns.countplot(y=genres, order=genres.value_counts().index)
plt.title("Movie Genre Distribution")
plt.xlabel("Count")
plt.ylabel("Genre")
plt.show()

In [ ]:
# Analyze and visualize the trend of ratings over time (by year)
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')

ratings['year'] = ratings['timestamp'].dt.year

ratings_per_year = ratings.groupby('year').size()

plt.figure()
ratings_per_year.plot()
plt.title("Ratings Over Time")
plt.xlabel("Year")
plt.ylabel("Number of Ratings")
plt.show()

# Content-Based Recommender with Metadata Features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#preproces Genres
#replaced '|' with spaces so TF-IDF can treat them as words

movies['genres_processed'] = movies['genres'].str.replace('|', ' ', regex=False)
# Preview
movies[['title', 'genres', 'genres_processed']].head()

In [ ]:
#Aggregate Tags for Each Movie

# Convert tags to string (avoid NaN issues)
tags['tag'] = tags['tag'].astype(str)

# Aggregate all tags belonging to the same movie
movie_tags = tags.groupby('movieId')['tag'].apply(lambda x: " ".join(x)).reset_index()

movie_tags.head()

In [ ]:
#Merge Tags with Movies

movies_with_tags = movies.merge(movie_tags, on='movieId', how='left')

# Replace missing tags with empty string
movies_with_tags['tag'] = movies_with_tags['tag'].fillna('')

movies_with_tags.head()

In [ ]:
#Create Combined Feature (Genres + Tags)

movies_with_tags['combined_features'] = (
    movies_with_tags['genres_processed'] + " " + movies_with_tags['tag']
)

movies_with_tags[['title','combined_features']].head()

In [ ]:
#TF-IDF Vectorization

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_with_tags['combined_features'])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

In [ ]:
# Cosine Similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

In [ ]:
# Building Movie Index Mapping

indices = pd.Series(movies_with_tags.index, index=movies_with_tags['title']).drop_duplicates()
indices.head()

In [ ]:
# 8. Recommendation Function
def recommend_movies(title, cosine_sim=cosine_sim):

    # Get movie index
    idx = indices.get(title)

    if idx is None:
        print("Movie not found in dataset")
        return

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Remove the movie itself
    sim_scores = sim_scores[1:11]

    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return movie titles
    return movies_with_tags['title'].iloc[movie_indices]

In [ ]:
import re
import ipywidgets as widgets
from IPython.display import display, clear_output

#word based search function
def search_movies(query):
    pattern = re.compile(r'\b' + re.escape(query.lower()))

    matches = [
        title for title in movies_with_tags['title']
        if pattern.search(title.lower())
    ]

    return matches[:20]

# UI widgets
search_box = widgets.Text(
    placeholder='Type a movie name (e.g. Toy)',
    description='Search:',
    layout=widgets.Layout(width='50%')
)

results = widgets.Select(
    options=[],
    rows=10,
    description='Movies:',
    layout=widgets.Layout(width='50%')
)

output = widgets.Output()

# Update results while typing
def update_results(change):
    query = change['new']

    if len(query) >= 2:
        results.options = search_movies(query)
    else:
        results.options = []

search_box.observe(update_results, names='value')

# Show recommendations
def show_recommendations(change):
    movie = change['new']
    with output:
        clear_output()

        if movie not in indices:
            print("Movie not found.")
            return

        print(f"\nTop 10 recommendations for: {movie}\n")

        recs = recommend_movies(movie)

        for i, r in enumerate(recs, 1):
            print(f"{i}. {r}")

results.observe(show_recommendations, names='value')
display(search_box)
display(results)
display(output)

# User-User Collaborative Filtering

In [ ]:
#creating User Item-Matrix
# Rows = users
# Columns = movies
# Values = ratings

user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

# Fill missing ratings with 0 (needed for cosine similarity)
user_item_filled = user_item_matrix.fillna(0)
user_item_filled.head(5)

In [ ]:
#Computing Cosine Similarity
# Cosine similarity between user vectors

user_vectors = user_item_filled.values

# Compute cosine similarity manually using numpy
dot_product = np.dot(user_vectors, user_vectors.T)

norms = np.linalg.norm(user_vectors, axis=1)

user_similarity = dot_product / (norms[:, None] * norms[None, :])

# Convert to DataFrame for easier indexing
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
user_similarity_df.head(5)

In [ ]:
# Predict Ratings using User-User CF
def predict_ratings(user_id):
    sim_scores = user_similarity_df.loc[user_id]
    weighted_ratings = np.dot(sim_scores, user_item_filled)
    sim_sum = np.abs(sim_scores).sum()

    if sim_sum == 0:
        predicted = np.zeros(len(user_item_matrix.columns))
    else:
        predicted = weighted_ratings / sim_sum

    return pd.Series(predicted, index=user_item_matrix.columns)


# Recommendation function
def recommend_movies_user(user_id, top_n=10):
    user_predictions = predict_ratings(user_id)
    user_predictions = user_predictions.sort_values(ascending=False)

    top_movie_ids = user_predictions.head(top_n).index

    recommendations = movies[movies['movieId'].isin(top_movie_ids)].copy()
    recommendations = recommendations[['movieId', 'title']]

    return recommendations

In [ ]:
# #Recommendation FUnction
# def recommend_movies_user(user_id, top_n=10):

#       user_predictions = predict_ratings(user_id)
#       user_predictions = user_predictions.sort_values(ascending=False)
#       top_movie_ids = user_predictions.head(top_n).index
#       recommendations = movies[movies['movieId'].isin(top_movie_ids)]
#       return recommendations[['title']]

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

user_box = widgets.IntText(
    value=1,
    description='User ID:'
)

button = widgets.Button(description="Recommend Movies")

output = widgets.Output()

def recommend_for_user(b):

    with output:
        clear_output()

        user_id = user_box.value

        print(f"\nTop recommendations for User {user_id}:\n")

        recs = recommend_movies_user(user_id)

        for i, title in enumerate(recs['title'], 1):
            print(f"{i}. {title}")

button.on_click(recommend_for_user)

display(user_box, button, output)

# Matrix Factorisation using SVD

In [ ]:
#Create User-Item Matrix\

user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)
# Fill missing values with 0
R = user_item_matrix.fillna(0).values

In [ ]:
#Apply Singular Value Decomposition
# Decompose matrix into latent factors

U, sigma, Vt = np.linalg.svd(R, full_matrices=False)
# Convert sigma to diagonal matrix
sigma = np.diag(sigma)

In [ ]:
#Reduce dimensionality
# Keep only top k features

k = 50
U_k = U[:, :k]
sigma_k = sigma[:k, :k]
Vt_k = Vt[:k, :]

In [ ]:
#Reconstruct predicted rating matrix
# Approximate the original matrix

R_pred = np.dot(np.dot(U_k, sigma_k), Vt_k)

predicted_ratings = pd.DataFrame(
    R_pred,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

In [ ]:
# Recommend movies for a user using SVD
def recommend_movies_svd(user_id, top_n=10):

    user_predictions = predicted_ratings.loc[user_id].copy()
    top_movie_ids = user_predictions.sort_values(ascending=False).head(top_n).index

    recommendations = movies[movies['movieId'].isin(top_movie_ids)].copy()
    recommendations = recommendations[['movieId', 'title']]

    return recommendations

import ipywidgets as widgets
from IPython.display import display, clear_output


user_box = widgets.IntText(
    value=1,
    description='User ID:'
)

button = widgets.Button(
    description="Recommend Movies",
    button_style='success'
)

output = widgets.Output()


def recommend_for_user(b):

    with output:
        clear_output()

        user_id = user_box.value

        if user_id not in user_item_matrix.index:
            print("User ID not found.")
            return

        print(f"\nTop recommendations for User {user_id} (Matrix Factorization):\n")

        recs = recommend_movies_svd(user_id)

        for i, title in enumerate(recs['title'], 1):
            print(f"{i}. {title}")


button.on_click(recommend_for_user)
display(user_box, button, output)

# Model Evaluation and Comparison

In [ ]:
# =========================================
# Model Evaluation (Fixed)
# =========================================

RELEVANCE_THRESHOLD = 4
K = 10
users = ratings['userId'].unique()[:50]

precision_content, recall_content = [], []
precision_cf, recall_cf = [], []
precision_svd, recall_svd = [], []


# Precision@K
def precision_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / k


# Recall@K
def recall_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    if len(relevant) == 0:
        return 0
    return hits / len(relevant)


# Evaluation loop
for user in users:
    user_data = ratings[ratings['userId'] == user]

    if len(user_data) < 5:
        continue

    # Relevant movies = movies the user liked
    relevant = user_data[user_data['rating'] >= RELEVANCE_THRESHOLD]['movieId'].tolist()

    if len(relevant) < 3:
        continue

    # ---------- Content-Based ----------
    try:
        liked_movies = relevant[:3]
        rec_ids = []

        for movie_id in liked_movies:
            title = movies.loc[movies['movieId'] == movie_id, 'title'].values
            if len(title) == 0:
                continue

            recs = recommend_movies(title[0])  # this returns titles
            ids = movies[movies['title'].isin(recs)]['movieId'].tolist()
            rec_ids.extend(ids)

        rec_ids = list(set(rec_ids))[:K]

        precision_content.append(precision_at_k(rec_ids, relevant, K))
        recall_content.append(recall_at_k(rec_ids, relevant, K))

    except:
        pass

    # ---------- Collaborative Filtering ----------
    try:
        recs = recommend_movies_user(user)   # returns movieId + title
        rec_ids = recs['movieId'].tolist()

        precision_cf.append(precision_at_k(rec_ids, relevant, K))
        recall_cf.append(recall_at_k(rec_ids, relevant, K))

    except:
        pass

    # ---------- Matrix Factorization ----------
    try:
        recs = recommend_movies_svd(user)    # returns movieId + title
        rec_ids = recs['movieId'].tolist()

        precision_svd.append(precision_at_k(rec_ids, relevant, K))
        recall_svd.append(recall_at_k(rec_ids, relevant, K))

    except:
        pass


# Compute averages
content_precision = np.mean(precision_content) if precision_content else 0
content_recall = np.mean(recall_content) if recall_content else 0

cf_precision = np.mean(precision_cf) if precision_cf else 0
cf_recall = np.mean(recall_cf) if recall_cf else 0

svd_precision = np.mean(precision_svd) if precision_svd else 0
svd_recall = np.mean(recall_svd) if recall_svd else 0


# =========================================
# RMSE
# =========================================

from sklearn.metrics import mean_squared_error

actual, predicted = [], []

for _, row in ratings.sample(5000).iterrows():
    user = row['userId']
    movie = row['movieId']

    if user in predicted_ratings.index and movie in predicted_ratings.index:
        pass

# SVD RMSE
actual, predicted = [], []

for _, row in ratings.sample(5000).iterrows():
    user = row['userId']
    movie = row['movieId']

    if user in predicted_ratings.index and movie in predicted_ratings.columns:
        actual.append(row['rating'])
        predicted.append(predicted_ratings.loc[user, movie])

rmse_svd = np.sqrt(mean_squared_error(actual, predicted))


# CF RMSE
def predict_rating_cf(user_id, movie_id):
    sim_scores = user_similarity_df.loc[user_id]
    movie_ratings = user_item_matrix[movie_id].fillna(0)

    weighted_sum = np.dot(sim_scores, movie_ratings)
    sim_sum = np.abs(sim_scores).sum()

    if sim_sum == 0:
        return 0

    return weighted_sum / sim_sum


actual_cf = []
predicted_cf = []

for _, row in ratings.sample(5000).iterrows():
    user = row['userId']
    movie = row['movieId']

    if movie in user_item_matrix.columns:
        pred = predict_rating_cf(user, movie)
        actual_cf.append(row['rating'])
        predicted_cf.append(pred)

rmse_cf = np.sqrt(mean_squared_error(actual_cf, predicted_cf))


# =========================================
# Comparison Table
# =========================================

comparison = pd.DataFrame({
    "Model": [
        "Content-Based",
        "Collaborative Filtering",
        "Matrix Factorization (SVD)"
    ],
    "RMSE": [
        np.nan,
        rmse_cf,
        rmse_svd
    ],
    "Precision@10": [
        content_precision,
        cf_precision,
        svd_precision
    ],
    "Recall@10": [
        content_recall,
        cf_recall,
        svd_recall
    ]
})

display(comparison)


# Precision / Recall plot
comparison.set_index("Model")[["Precision@10", "Recall@10"]].plot(kind="bar", figsize=(8, 5))
plt.title("Recommendation Model Comparison")
plt.ylabel("Score")
plt.tight_layout()
plt.show()


# RMSE plot
comparison.dropna(subset=["RMSE"]).set_index("Model")["RMSE"].plot(kind="bar", figsize=(6, 4))
plt.title("RMSE Comparison")
plt.ylabel("RMSE")
plt.tight_layout()
plt.show()

## Model Evaluation and Comparison Summary

Based on the evaluation metrics, here's a summary and comparison of the three recommendation models:

*   **Matrix Factorization (SVD)**: This model demonstrates the best performance overall, achieving the lowest RMSE of **2.014** and the highest Precision@10 (**0.664**) and Recall@10 (**0.163**). This suggests it is the most accurate at predicting ratings and providing relevant recommendations among the evaluated models.

*   **Collaborative Filtering (User-User)**: This model shows good performance, especially in terms of Precision@10 (**0.480**) and Recall@10 (**0.116**), but its RMSE of **3.057** indicates that its rating predictions are less accurate than SVD. It relies on user similarity to make recommendations.

*   **Content-Based**: This model has a significantly lower Precision@10 (**0.094**) and Recall@10 (**0.015**), and its RMSE is not available in this comparison. This suggests that for the chosen evaluation strategy and dataset, the content-based approach (using genres and tags) is less effective than the other two models in providing relevant recommendations.

In [ ]:
# import pickle

# model_data = {
#     "tfidf": tfidf,
#     "tfidf_matrix": tfidf_matrix,
#     "movies": movies_with_tags,
#     "indices": indices
# }

# with open("model.pkl", "wb") as f:
#     pickle.dump(model_data, f)

# print("Model saved successfully!")

In [ ]:
# from google.colab import files
# files.download("model.pkl")